In [1]:
import pandas as pd
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import glob
import seaborn as sns  # Optional for aesthetics
from scipy.stats import linregress
import datetime
from concave_hull import concave_hull, concave_hull_indexes
from sklearn.cluster import DBSCAN
import numpy as np
import fda as skfda
from scipy.linalg import pinv

data_path  = '/Users/epauthen/Documents/Database/Antarctic_Sea_Ice/v3p0_monthly_mean/'
shape_path = '/Users/epauthen/Documents/Database/Antarctic_Sea_Ice/v3p0_monthly_mean_shape15/'


In [2]:
def find_connected_areas(points, epsilon, min_samples):
    # Convert the points to a NumPy array
    data = np.array(points)

    # Apply DBSCAN clustering
    dbscan = DBSCAN(eps=epsilon, min_samples=min_samples)
    labels = dbscan.fit_predict(data)

    # Create a dictionary to store connected areas
    connected_areas = {}
    for i, label in enumerate(labels):
        if label != -1:  # Ignore noise points
            if label not in connected_areas:
                connected_areas[label] = []
            connected_areas[label].append(tuple(data[i]))

    # Convert dictionary values to lists
    connected_areas = list(connected_areas.values())

    return connected_areas

In [41]:
file_list = glob.glob(data_path+ '*.nc')
#n = 0
for n in np.arange(len(file_list)):
    dsy = xr.open_dataset(file_list[n])
    for tt in np.arange(12):
        ds = dsy.isel(time = tt)
        ice_conc_mask = np.isnan(ds['ice_conc'])
        mask_dataarray = xr.DataArray(ice_conc_mask, coords=ds['ice_conc'].coords)
        mask_dataarray.name = 'ice_conc_mask'
        mask_dataarray_filtered = mask_dataarray.where(ds['lat'] <= -60, 0)
        ds = ds.assign(mask=mask_dataarray_filtered[:,:])

        #Get x,y points for land mask and SIC15
        ind_mask = np.where(ds.mask == 1)
        threshold_ice_conc = 15
        ind_SIC15 = np.where(ds.ice_conc > threshold_ice_conc)

        # Get the corresponding coordinates using the indices
        x_mask = ds.xc[ind_mask[1]].values
        y_mask = ds.yc[ind_mask[0]].values
        x_15 = ds.xc[ind_SIC15[1]].values
        y_15 = ds.yc[ind_SIC15[0]].values    
        x = np.concatenate((x_mask, x_15))
        y = np.concatenate((y_mask, y_15))
        points = np.vstack((x,y)).T

        epsilon = 25  # Maximum distance between points in the same neighborhood
        min_samples = 2  # Minimum number of samples in a neighborhood

        connected_areas = find_connected_areas(points, epsilon, min_samples)

        #Find length maximum of clusters and apply the hull on it 
        ca_size = np.array(connected_areas,dtype=object).size
        area_size = list(np.arange(ca_size))
        for i,area, in zip(np.arange(ca_size),connected_areas):
            area_size[i] = np.size(area)
        ind_max = np.argmax(area_size)
        V = connected_areas[ind_max]

        #Concave hull of the coastline + main sea ice shapes
        a_hull = np.array(concave_hull(V,concavity = .1))

        #Recenter to the Wester Peninsula
        x = a_hull[:,0]
        y = a_hull[:,1]
        target_point = np.array([-2550, 1300])
        distances = np.sqrt((x - target_point[0])**2 + (y - target_point[1])**2)
        min_distance_index = np.argmin(distances)
        Cx = np.concatenate((x[min_distance_index:], x[:min_distance_index]))
        Cy = np.concatenate((y[min_distance_index:], y[:min_distance_index]))
        
        #Save Netcdf
    #    ds = ds.assign_coords(time=("time", ds.time.data))
        Cx = xr.DataArray(Cx, dims=('Cx'), name='Cx')
        Cy = xr.DataArray(Cy, dims=('Cy'), name='Cy')
        ds = ds.assign(C15x=Cx,C15y=Cy)
        ds = ds.expand_dims('time')
        variables_to_keep = ['time', 'C15x', 'C15y']
        ds = ds.drop_vars([var for var in ds.variables if var not in variables_to_keep])
        ds.to_netcdf(shape_path + 'Shape15_' + str(ds.time[0].data)[0:7] + '.nc')


IndexError: index 3 is out of bounds for axis 0 with size 3